# PPDONet-Stan: Velocidad Radial

Entrenamiento y evaluacion del modelo PPDONet para el mapa de velocidad radial `v_r`.

Los parametros fisicos (`ALPHA`, `PLANETMASS`, `ASPECTRATIO`) alimentan la red branch. Las coordenadas del mapa (`r`, `theta`) alimentan la red trunk mediante la transformacion periodica `(r_scaled, sin(theta), cos(theta))`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

from ppdonet_common import *

device = get_device()
print_device_info(device)


## Datos

In [ ]:
train_dir, test_dir = download_planet2disk_data()
print("Train dir:", train_dir)
print("Test dir :", test_dir)

ds_sigma_tr, ds_vr_tr, ds_vtheta_tr = open_outputs(train_dir)
ds_sigma_te, ds_vr_te, ds_vtheta_te = open_outputs(test_dir)

params_tr, where_tr = find_and_load_params(train_dir, device)
params_te, where_te = find_and_load_params(test_dir, device)
print(where_tr)
print("params_tr:", params_tr.shape, params_tr.device)


In [ ]:
coords_tf, r_vals, th_vals = build_coords(ds_sigma_tr, device)
print("coords_tf:", coords_tf.shape, coords_tf.device)


## Dataset y loaders

In [ ]:
vr_var = list(ds_vr_tr.data_vars)[0]

NPOINTS_SAMPLE = 435483
BATCH_SIZE = 1

train_vr_loader, test_vr_loader = make_field_loaders(
    ds_vr_tr,
    ds_vr_te,
    params_tr,
    params_te,
    coords_tf,
    vr_var,
    device,
    log_target=False,
    subtract_background=True,
    n_points_sample=NPOINTS_SAMPLE,
    batch_size=BATCH_SIZE,
)

u0, y0, t0 = next(iter(train_vr_loader))
print("sample shapes (u, y, target):", u0.shape, y0.shape, t0.shape)
print("devices:", u0.device, y0.device, t0.device)


## Entrenamiento

In [ ]:
EPOCHS = 1500
LR = 1e-4
CHECKPOINT = "modelo_vr_Stan.pt"

model_vr = build_ppdonet(act="stan", stan_beta=0.1, stan_positive_beta=True)
hist_vr, model_vr = train_model(
    model_vr,
    train_vr_loader,
    test_vr_loader,
    device,
    epochs=EPOCHS,
    lr=LR,
)
save_model(model_vr, CHECKPOINT)


## Evaluacion

In [ ]:
CHECKPOINT = "modelo_vr_Stan.pt"
model_vr = load_model(CHECKPOINT, device)

mse_vr, r2_vr = evaluate_model(model_vr, test_vr_loader, device)
print(f"v_r: MSE={mse_vr:.4e}, R2={r2_vr:.4f}")


## Ejemplo de test

In [ ]:
idx_example = 0

pred_vr_map, true_vr_map, mse_vr_ex, r2_vr_ex = predict_single_field(
    model_vr,
    params_te,
    coords_tf,
    ds_vr_te,
    vr_var,
    r_vals,
    th_vals,
    device,
    idx_example=idx_example,
    log_target=False,
    subtract_background=True,
    fargo_kepler_delta=False,
)

print(f"v_r ejemplo {idx_example}: MSE={mse_vr_ex:.4e}, R2={r2_vr_ex:.4f}")


In [ ]:
plot_field_maps(
    pred_vr_map,
    true_vr_map,
    r_vals,
    th_vals,
    title=f"v_r (delta) - Test idx={idx_example}",
    cmap="viridis",
    robust=False,
)
